In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/tax_project"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/tax_project/audit_records.csv,audit_records.csv,114996,1781599601000
dbfs:/Volumes/workspace/default/tax_project/tax_filings.csv,tax_filings.csv,451746,1781599602000
dbfs:/Volumes/workspace/default/tax_project/tax_transactions.csv,tax_transactions.csv,5068639,1781599606000


In [0]:
transactions = spark.read.csv(
"/Volumes/workspace/default/tax_project/tax_transactions.csv",
header=True,
inferSchema=True
)

filings = spark.read.csv(
"/Volumes/workspace/default/tax_project/tax_filings.csv",
header=True,
inferSchema=True
)

audits = spark.read.csv(
"/Volumes/workspace/default/tax_project/audit_records.csv",
header=True,
inferSchema=True
)

In [0]:
print("Transactions:", transactions.count())
print("Filings:", filings.count())
print("Audits:", audits.count())

Transactions: 100000
Filings: 10000
Audits: 5000


In [0]:
display(transactions.limit(10))

invoice_id,country,tax_type,revenue,tax_amount,invoice_date
INV000001,Singapore,VAT,223508,26268.65,2025-12-26
INV000002,USA,VAT,289562,22861.77,2025-12-21
INV000003,UK,VAT,478426,85316.96,2025-06-23
INV000004,USA,GST,469260,56012.26,2025-07-03
INV000005,USA,Corporate Tax,303537,27656.89,2024-02-07
INV000006,Germany,VAT,422935,55803.99,2025-10-19
INV000007,UK,Sales Tax,58044,6432.73,2024-11-11
INV000008,UK,VAT,322414,59119.38,2024-08-13
INV000009,UK,VAT,332484,32472.99,2025-02-15
INV000010,USA,VAT,482545,28286.14,2024-09-09


### **Tax Exposure Analysis**

In [0]:
from pyspark.sql.functions import sum

tax_exposure = (
    transactions
    .groupBy("country")
    .agg(
        sum("tax_amount").alias("total_tax_exposure")
    )
)

display(tax_exposure)

country,total_tax_exposure
USA,6.2854327131E8
Singapore,6.244265265999999E8
India,6.251842488800006E8
UK,6.170519809600004E8
Germany,6.281190497699991E8


### Filing Delay Analysis

In [0]:
from pyspark.sql.functions import *

filings = filings.withColumn(
    "due_date",
    to_date("due_date")
)

filings = filings.withColumn(
    "filing_date",
    to_date("filing_date")
)

filings = filings.withColumn(
    "delay_days",
    datediff(
        col("filing_date"),
        col("due_date")
    )
)

delay_analysis = (
    filings
    .groupBy("country")
    .agg(
        avg("delay_days")
        .alias("avg_delay_days")
    )
)

display(delay_analysis)

country,avg_delay_days
USA,12.67508667657256
Singapore,12.492063492063492
India,12.53495145631068
UK,12.26923076923077
Germany,12.43031536113937


### Compliance %

In [0]:
filings = filings.withColumn(
    "on_time",
    when(col("delay_days") <= 0,1)
    .otherwise(0)
)

compliance_rate = (
    filings
    .groupBy("country")
    .agg(
        (
            avg("on_time")*100
        ).alias("compliance_percent")
    )
)

display(compliance_rate)

country,compliance_percent
USA,16.443784051510647
Singapore,16.180235535074246
India,17.33009708737864
UK,16.98301698301698
Germany,16.531027466937946


### Audit Risk KPI

In [0]:
audit_risk = (
    audits
    .groupBy("country")
    .agg(
        avg("issue_count")
        .alias("avg_audit_issues")
    )
)

display(audit_risk)

country,avg_audit_issues
USA,2.9989733059548254
Singapore,2.997067448680352
India,2.931137724550898
UK,3.013861386138614
Germany,2.8486377396569122


### Spark SQL

In [0]:
tax_exposure.createOrReplaceTempView("tax_exposure")
compliance_rate.createOrReplaceTempView("compliance_rate")
audit_risk.createOrReplaceTempView("audit_risk")

In [0]:
%sql
SELECT
country,
total_tax_exposure
FROM tax_exposure
ORDER BY total_tax_exposure DESC

country,total_tax_exposure
USA,6.2854327131E8
Germany,6.281190497699991E8
India,6.251842488800006E8
Singapore,6.244265265999999E8
UK,6.170519809600004E8


### Final KPI Table

In [0]:
final_dashboard = (
    tax_exposure
    .join(compliance_rate,"country")
    .join(audit_risk,"country")
)

display(final_dashboard)

country,total_tax_exposure,compliance_percent,avg_audit_issues
USA,6.2854327131E8,16.443784051510647,2.9989733059548254
Singapore,6.244265265999999E8,16.180235535074246,2.997067448680352
India,6.251842488800006E8,17.33009708737864,2.931137724550898
UK,6.170519809600004E8,16.98301698301698,3.013861386138614
Germany,6.281190497699991E8,16.531027466937946,2.8486377396569122


### Add Risk Level

In [0]:
from pyspark.sql.functions import *

final_dashboard = final_dashboard.withColumn(
    "risk_level",
    when(col("compliance_percent") < 16.5, "High")
    .when(col("compliance_percent") < 17, "Medium")
    .otherwise("Low")
)

display(final_dashboard)

country,total_tax_exposure,compliance_percent,avg_audit_issues,risk_level
USA,6.2854327131E8,16.443784051510647,2.9989733059548254,High
Singapore,6.244265265999999E8,16.180235535074246,2.997067448680352,High
India,6.251842488800006E8,17.33009708737864,2.931137724550898,Low
UK,6.170519809600004E8,16.98301698301698,3.013861386138614,Medium
Germany,6.281190497699991E8,16.531027466937946,2.8486377396569122,Medium


In [0]:
final_dashboard.write.mode("overwrite").saveAsTable(
    "workspace.default.tax_risk_dashboard"
)

In [0]:
%sql
SELECT *
FROM workspace.default.tax_risk_dashboard
ORDER BY total_tax_exposure DESC;

country,total_tax_exposure,compliance_percent,avg_audit_issues,risk_level
USA,6.2854327131E8,16.443784051510647,2.9989733059548254,High
Germany,6.281190497699991E8,16.531027466937946,2.8486377396569122,Medium
India,6.251842488800006E8,17.33009708737864,2.931137724550898,Low
Singapore,6.244265265999999E8,16.180235535074246,2.997067448680352,High
UK,6.170519809600004E8,16.98301698301698,3.013861386138614,Medium
